# Partie A — Cavité entraînée à $Re = 100$ : solveur de référence vérifié

Énoncé, Partie A : calculer l'écoulement stationnaire à $Re = 100$, le valider (littérature et/ou autre solveur validé) et justifier que la grille suffit.

1. Vérification du solveur de Poisson (gradient conjugué sans matrice) sur des solutions connues.
2. Écoulement stationnaire à $Re = 100$ avec les deux méthodes : M1 (projection) et M2 ($\psi$–$\omega$), $N = 32, 64, 128$.
3. Validation : M1, M2 et Ghia et al. (1982).
4. Justification de la grille.

Adimensionnement $L = U = \rho = 1$, $\nu = 1/Re$. Grille colocalisée aux nœuds, $h = 1/N$. Euler explicite, advection et diffusion centrées,
$\Delta t$ = limites de stabilité du cours (marge 0.8). Arrêt quand le résidu stationnaire $\|R\|$ est divisé par $10^3$ depuis $t = 30$ (même critère
que la Partie B ; $R$ sur $u, v$ pour M1, sur $\omega$ pour M2).

**Les choix intermédiaires et les difficultés (Neumann d'ordre 1 abandonnée, erreur de projection de M1 en $\Delta t$, tolérance du gradient conjugué,
$P$ et les coins du couvercle) sont documentés dans [`Doc/partie_A_limites.md`](../Doc/partie_A_limites.md).** Ce notebook ne garde que les résultats.

In [ ]:
import os
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"   # défaut de l'environnement CFD sur cette machine : à fixer avant numpy
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
FIG, RES, REF = ROOT / "figures", ROOT / "results", ROOT / "ref_data"
FIG.mkdir(exist_ok=True)

from cavity.common import solve_poisson
from cavity.projection import run_projection
from cavity.psi_omega import run_psi_omega

RE, GRIDS = 100, (32, 64, 128)
CG_TOL, EPS, T_REF, T_MAX = 1e-8, 1e-3, 30.0, 1000.0
RUNNERS = {"M1": run_projection, "M2": run_psi_omega}


def run_path(method, n):
    # Mêmes fichiers que la Partie B (même critère d'arrêt, Re = 100)
    return RES / f"B_{method}_Re{RE}_N{n}_tol{CG_TOL:.0e}_res{EPS:.0e}_tref{T_REF:g}.npz"


def get_run(method, n):
    # Relit le calcul s'il existe, sinon le lance (quelques secondes à N = 32, ~20 min pour M1 à N = 128)
    path = run_path(method, n)
    if not path.exists():
        run = RUNNERS[method](n, RE, "centered", cg_tol=CG_TOL, residual_tol=EPS, t_ref=T_REF, t_max=T_MAX)
        np.savez(path, **{key: np.asarray(val) for key, val in run.items()})
    data = np.load(path)
    return {key: (data[key].item() if data[key].ndim == 0 else data[key]) for key in data.files}


def centerlines(run):
    # u(0.5, y) sur la colonne i = N/2 et v(x, 0.5) sur la ligne j = N/2
    n = run["N"]
    return run["u"][n // 2, :], run["v"][:, n // 2]


def order(errors):
    # Ordre observé entre deux grilles successives (h divisé par 2) : log2(e_N / e_2N)
    e = np.asarray(errors, float)
    return np.log2(e[:-1] / e[1:])

## 1. Vérification du solveur de Poisson

**Ce qu'on fait.** On résout $\Delta\phi = -2\pi^2\phi$ sur le carré unité avec deux solutions connues, une par condition aux limites utilisée :
- Dirichlet, $\phi = \sin(\pi x)\sin(\pi y)$ ($\phi = 0$ aux parois) : c'est l'équation $\Delta\psi = -\omega$ de M2 ;
- Neumann, $\phi = \cos(\pi x)\cos(\pi y)$ ($\partial\phi/\partial n = 0$) : c'est l'équation de pression de M1. Neumann d'ordre 2 par nœuds fantômes
  (miroir $\phi_{-1} = \phi_1$), moyenne du second membre retirée (compatibilité), moyenne de $\phi$ nulle à chaque itération (unicité).

**Ce qu'on attend.** Erreur d'ordre 2 (laplacien centré à 5 points, cours §2.3) ; $O(N)$ itérations du gradient conjugué (cours §4.8).

In [ ]:
rows = []
for n in (16, 32, 64):
    h = 1.0 / n
    x = np.linspace(0.0, 1.0, n + 1)
    X, Y = np.meshgrid(x, x, indexing="ij")                    # [i, j] <-> (x, y)
    phi_d = np.sin(np.pi * X) * np.sin(np.pi * Y)
    phi_n = np.cos(np.pi * X) * np.cos(np.pi * Y)
    sol_d, it_d = solve_poisson((-2 * np.pi**2 * phi_d)[1:-1, 1:-1], h, "dirichlet", tol=1e-10)
    sol_n, it_n = solve_poisson(-2 * np.pi**2 * phi_n, h, "neumann_ghost", tol=1e-10)
    rows.append((n, np.abs(sol_d - phi_d).max(), it_d, np.abs(sol_n - phi_n).max(), it_n))

print(f"{'N':>4} | {'erreur Dirichlet':>16} | {'erreur Neumann':>14} {'itérations GC':>14}")
for n, ed, it_d, en, it_n in rows:
    print(f"{n:4d} | {ed:16.2e} | {en:14.2e} {it_n:14d}")
print("ordre observé : Dirichlet", np.round(order([r[1] for r in rows]), 2), "  Neumann", np.round(order([r[3] for r in rows]), 2))

**Lecture.** Ordre **2.00** pour les deux conditions aux limites, et le nombre d'itérations double quand $N$ double ($O(N)$). Le solveur de Poisson
est vérifié. (En Dirichlet, $\sin\sin$ est un vecteur propre du laplacien discret : le gradient conjugué converge en 1 itération.)

## 2. Écoulement stationnaire à $Re = 100$

**Ce qu'on fait.** M1 et M2 depuis le repos (couvercle $U_t = 1 - e^{-t}$) jusqu'au stationnaire, $N = 32, 64, 128$. Lignes de courant à $N = 128$ et
diagnostics de l'énoncé, $K(t)$ et $P(t)$.

**Ce qu'on attend.** Un tourbillon principal décalé vers le haut à droite (sens de l'entraînement), deux petits tourbillons dans les coins du bas ;
$K$ et $P$ qui suivent la rampe du couvercle puis atteignent un palier.

In [ ]:
runs = {(m, n): get_run(m, n) for m in ("M1", "M2") for n in GRIDS}


def streamfunction(run):
    # ψ : stockée par M2 ; reconstruite pour M1 par Δψ = -ω, ψ = 0 aux parois
    if "psi" in run:
        return run["psi"]
    u, v, h = run["u"], run["v"], run["h"]
    omega = ((v[2:, 1:-1] - v[:-2, 1:-1]) - (u[1:-1, 2:] - u[1:-1, :-2])) / (2.0 * h)
    return solve_poisson(-omega, h, "dirichlet", tol=1e-12)[0]


fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, m in zip(axes[:2], ("M1", "M2")):
    r = runs[(m, 128)]
    X, Y = np.meshgrid(r["x"], r["y"], indexing="ij")
    psi = streamfunction(r)
    ax.contour(X, Y, psi, levels=[-0.1, -0.09, -0.07, -0.05, -0.03, -0.01, -1e-3, -1e-4], colors="k", linewidths=0.6)
    ax.contour(X, Y, psi, levels=[1e-7, 1e-6, 5e-6, 1e-5], colors="r", linewidths=0.5)
    ax.set_aspect("equal"); ax.set_title(f"{m}, N = 128 : lignes de courant")
for m, ls in (("M1", "-"), ("M2", "--")):
    r = runs[(m, 128)]
    keep = r["t"] <= 30
    axes[2].plot(r["t"][keep], r["K"][keep], ls=ls, label=m)
    axes[3].plot(r["t"][keep], r["P"][keep], ls=ls, label=m)
axes[2].set_title("$K(t)$, N = 128"); axes[3].set_title("$P(t)$, N = 128")
for ax in axes[2:]:
    ax.set_xlabel("$t$"); ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(FIG / "A_flow_K_P.png", dpi=150)
plt.show()

print(f"{'méth.':>5} {'N':>4} | {'Δt':>8} {'arrêt t':>8} {'calcul (s)':>10} | {'K':>8} {'P':>8}")
for (m, n), r in runs.items():
    print(f"{m:>5} {n:4d} | {r['dt']:8.1e} {r['t_final']:8.0f} {r['wall_time']:10.1f} | {r['K'][-1]:8.5f} {r['P'][-1]:8.5f}")

**Lecture.**
- Tourbillon principal centré en $(0.62, 0.73)$, deux tourbillons de coin en bas, comme chez Ghia ; M1 et M2 donnent la même image.
- $K$ et $P$ suivent la rampe du couvercle et atteignent leur palier vers $t \approx 8$ : à $Re = 100$, le transitoire est un simple amortissement visqueux.
  (M1 à $N = 128$ va jusqu'au garde-fou $t = 1000$ : stationnaire avant $t = 30$, son résidu ne peut plus être divisé par $10^3$ ; le résultat est juste.)
- **$K$ converge** avec $N$ (M2 : 0.0339, 0.0344, 0.0344). **$P$ ne converge pas** (0.14, 0.17, 0.20 pour M2) : il croît comme $\ln N$ à cause de la
  singularité des deux coins du couvercle (la vitesse y saute de $U$ à 0). On ne compare donc les $P$ qu'à $N$ fixé.

## 3. Validation : M1, M2 et Ghia et al. (1982)

**Ce qu'on fait.** Profils $u(0.5, y)$ et $v(x, 0.5)$ des deux méthodes à $N = 128$ (la grille de Ghia, 129² points), avec les points de Ghia
(Tables I et II, `ref_data/`). Écart maximal à Ghia sur ses points intérieurs, et écart entre nos deux méthodes, pour les trois grilles.

**Ce qu'on attend.** Les deux méthodes discrétisent le même problème par des voies différentes (pression ou fonction de courant) : si elles convergent
vers la même solution, et que cette solution est celle de Ghia, le solveur est validé.

In [ ]:
ghia_u = np.loadtxt(REF / "ghia1982_u_vertical_centerline.csv", delimiter=",")   # colonne 1 : Re = 100
ghia_v = np.loadtxt(REF / "ghia1982_v_horizontal_centerline.csv", delimiter=",")
in_u, in_v = (ghia_u[:, 0] > 0) & (ghia_u[:, 0] < 1), (ghia_v[:, 0] > 0) & (ghia_v[:, 0] < 1)   # parois exclues (imposées)


def ghia_error(run):
    u_c, v_c = centerlines(run)
    return (np.abs(np.interp(ghia_u[in_u, 0], run["y"], u_c) - ghia_u[in_u, 1]).max(),
            np.abs(np.interp(ghia_v[in_v, 0], run["x"], v_c) - ghia_v[in_v, 1]).max())


fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for m, ls in (("M1", "-"), ("M2", "--")):
    r = runs[(m, 128)]
    u_c, v_c = centerlines(r)
    axes[0].plot(u_c, r["y"], ls=ls, label=f"{m}, N = 128")
    axes[1].plot(r["x"], v_c, ls=ls, label=f"{m}, N = 128")
axes[0].plot(ghia_u[:, 1], ghia_u[:, 0], "ko", mfc="none", label="Ghia et al. (1982)")
axes[1].plot(ghia_v[:, 0], ghia_v[:, 1], "ko", mfc="none", label="Ghia et al. (1982)")
axes[0].set_xlabel("$u(0.5, y)$"); axes[0].set_ylabel("$y$"); axes[0].set_title("Ligne verticale médiane")
axes[1].set_xlabel("$x$"); axes[1].set_ylabel("$v(x, 0.5)$"); axes[1].set_title("Ligne horizontale médiane")
for ax in axes:
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "A_profiles_Ghia.png", dpi=150)
plt.show()

print("max|Δu| / max|Δv| sur les lignes médianes")
print(f"{'N':>4} | {'M1 - Ghia':>15} | {'M2 - Ghia':>15} | {'M1 - M2':>15}")
for n in GRIDS:
    r1, r2 = runs[("M1", n)], runs[("M2", n)]
    d12 = [np.abs(a - b).max() for a, b in zip(centerlines(r1), centerlines(r2))]
    e1, e2 = ghia_error(r1), ghia_error(r2)
    print(f"{n:4d} | {e1[0]:6.4f} / {e1[1]:6.4f} | {e2[0]:6.4f} / {e2[1]:6.4f} | {d12[0]:6.4f} / {d12[1]:6.4f}")

**Lecture.**
- À $N = 128$, M1, M2 et Ghia se superposent à l'échelle du tracé : écart maximal à Ghia $\le 0.012$ pour les deux méthodes.
- **L'écart entre M1 et M2 diminue quand on raffine** (0.028, 0.011, 0.004) : les deux méthodes convergent vers **la même solution**, qui est celle de Ghia.
  C'est la validation par un second solveur demandée par l'énoncé.
- L'écart de M2 à Ghia ne diminue plus entre $N = 64$ et $128$ (0.003 puis 0.004 en $u$) alors que M2 converge (§4) : on atteint l'erreur propre de Ghia,
  qui a calculé sur 129² points.

## 4. La grille est-elle suffisante ?

**Ce qu'on fait.** Écart entre grilles successives, $\max|u_N - u_{2N}|$ sur les nœuds communs des lignes médianes : il ne dépend d'aucune référence et
donne l'ordre observé de chaque méthode.

**Ce qu'on attend.** Ordre 2 (schémas centrés d'ordre 2).

In [ ]:
gaps = {m: np.array([[np.abs(a - b[::2]).max() for a, b in zip(centerlines(runs[(m, n)]), centerlines(runs[(m, 2 * n)]))]
                     for n in GRIDS[:-1]]) for m in ("M1", "M2")}
h = 1.0 / np.array(GRIDS[:-1], float)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for m, mk in (("M1", "o-"), ("M2", "s--")):
    ax.loglog(h, gaps[m].max(axis=1), mk, label=f"{m} : ordre {order(gaps[m].max(axis=1))[0]:.2f}")
ax.loglog(h, gaps["M2"].max(axis=1)[0] * (h / h[0])**2, "k:", lw=0.8, label="pente 2")
ax.set_xlabel("$h$ (grille grossière de la paire)"); ax.set_ylabel(r"$\max|\cdot_N - \cdot_{2N}|$")
ax.set_title("Écart entre grilles successives, Re = 100"); ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "A_grid_convergence.png", dpi=150)
plt.show()

for m in ("M1", "M2"):
    print(f"{m} : écarts N=32→64, 64→128 (u, v) = {np.round(gaps[m], 4).tolist()}   ordre u {order(gaps[m][:, 0])[0]:.2f}, v {order(gaps[m][:, 1])[0]:.2f}")

**Lecture.**
- **M2 converge à l'ordre 2** (2.07 en $u$, 2.02 en $v$) : à $N = 128$, il est à $\sim 10^{-3}$ de sa limite. Les coins du couvercle ne dégradent pas l'ordre des profils.
- **M1 converge à l'ordre 1 seulement avec le $\Delta t$ de stabilité.** M1 est d'ordre 2 en espace (vérifié à $\Delta t$ fixe), mais sa solution stationnaire
  dépend aussi de $\Delta t$ (projection non incrémentale du cours) ; or $\Delta t$ change avec $N$. Détail : `Doc/partie_A_limites.md` §3.
- **$N = 128$ suffit** : les deux méthodes y sont à $0.004$ l'une de l'autre et à $\le 0.012$ de Ghia, M2 est à $\sim 10^{-3}$ de sa limite, et chaque
  doublement de $N$ coûte environ 16 fois plus. C'est aussi la grille de Ghia.

## Bilan de la Partie A

- Poisson vérifié à l'ordre 2 (Dirichlet et Neumann à nœuds fantômes).
- À $Re = 100$, M1 et M2 convergent vers la même solution stationnaire, celle de Ghia : écart M1 − M2 de $0.004$ à $N = 128$, écart à Ghia $\le 0.012$.
- $N = 128$ suffit ; $K$ converge, $P$ croît en $\ln N$ (coins), donc on ne compare les $P$ qu'à $N$ fixé.